# Final PTB-XL Evaluation (GPU / Google Drive)

Notebook version of `eval_final.py` for GPU/Colab runs. It keeps the same fair-evaluation rules: threshold is selected only on val, test is used once for final metrics, no TTA and no threshold search on test.

The notebook auto-mounts Google Drive in Colab, locates the project folder, and evaluates the enabled models from `MODELS_CONFIG`.


In [1]:
from pathlib import Path
import os
import sys

# If Colab cannot auto-detect your project folder, set this manually, for example:
# PROJECT_ROOT_OVERRIDE = "/content/drive/MyDrive/ecg-diploma-main 3"
PROJECT_ROOT_OVERRIDE = None

def _running_in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

def _mount_drive_if_colab():
    if not _running_in_colab():
        return
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")

def _find_project_root():
    if PROJECT_ROOT_OVERRIDE:
        root = Path(PROJECT_ROOT_OVERRIDE).expanduser()
        if not root.exists():
            raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE does not exist: {root}")
        return root

    candidates = [
        Path.cwd(),
        Path("/content/drive/MyDrive/ecg-diploma-main 3"),
        Path("/content/drive/MyDrive/ecg-diploma-main"),
        Path("/content/drive/MyDrive/ecg-multigraph-lab"),
        Path("/content/drive/MyDrive/train_d/ecg-diploma"),
    ]
    for root in candidates:
        if (root / "eval_final.py").exists() and (root / "src").exists():
            return root

    drive_root = Path("/content/drive/MyDrive")
    if drive_root.exists():
        for pattern in ("**/eval_final.py", "**/src/models/model_Damir/model_b_attention.py"):
            for hit in drive_root.glob(pattern):
                root = hit.parent if hit.name == "eval_final.py" else hit.parents[3]
                if (root / "src").exists() and (root / "results").exists():
                    return root

    raise FileNotFoundError("Could not find project root. Set PROJECT_ROOT_OVERRIDE in this cell.")

_mount_drive_if_colab()
PROJECT_ROOT = _find_project_root().resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data_preprocessed" / "ptbxl_sota_100hz_diagnostic_superclass.npz"
RESULTS_CSV = PROJECT_ROOT / "results" / "eval_final.csv"

# Safe defaults. Increase on a stronger GPU; lower RetNet if CUDA OOM happens.
DEFAULT_BATCH_SIZE = 64
DEVICE = "auto"
REQUIRE_GPU = True

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH exists:", DATA_PATH.exists(), DATA_PATH)
print("RESULTS_CSV:", RESULTS_CSV)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/ecg-diploma-main 3
DATA_PATH exists: True /content/drive/MyDrive/ecg-diploma-main 3/data_preprocessed/ptbxl_sota_100hz_diagnostic_superclass.npz
RESULTS_CSV: /content/drive/MyDrive/ecg-diploma-main 3/results/eval_final.csv


In [2]:
import csv
import importlib
import inspect
import os
import warnings
from typing import Dict

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score

warnings.filterwarnings("ignore")

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))


torch: 2.6.0+cu124
cuda available: True
cuda device: Tesla T4


In [3]:
import importlib as _importlib
import eval_final as _eval_final

_eval_final = _importlib.reload(_eval_final)
MODELS_CONFIG = _eval_final.MODELS_CONFIG
CLASS_NAMES = _eval_final.CLASS_NAMES

enabled = [name for name, cfg in MODELS_CONFIG.items() if cfg["enabled"]]
print("Enabled models:", enabled)


Enabled models: ['LeadWise_GNN', 'InceptionTime', 'RetNet']


In [4]:
def resolve_device(device: str = "auto") -> str:
    if device == "auto":
        if torch.cuda.is_available():
            return "cuda"
        if REQUIRE_GPU:
            raise RuntimeError("GPU is required for this notebook. In Colab: Runtime -> Change runtime type -> GPU, then rerun from the top.")
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return "mps"
        return "cpu"
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("DEVICE='cuda' was requested, but CUDA is not available.")
    return device


def load_data(data_path: str):
    data = np.load(data_path, allow_pickle=True)

    def get(keys):
        for k in keys:
            if k in data:
                return data[k]
        raise KeyError(f"Missing all keys: {keys}")

    if all(k in data for k in ["X_val", "y_val", "X_test", "y_test"]):
        X_val  = get(["X_val",  "x_val"])
        y_val  = get(["y_val",  "Y_val"])
        X_test = get(["X_test", "x_test"])
        y_test = get(["y_test", "Y_test"])
    elif all(k in data for k in ["signals", "labels", "splits"]):
        signals = data["signals"]
        labels  = data["labels"]
        splits  = data["splits"]

        if splits.dtype.kind in {"S", "O", "U"}:
            splits = splits.astype(str)
            val_mask = splits == "val"
            test_mask = splits == "test"
        else:
            val_mask = splits == 9
            test_mask = splits == 10

        X_val  = signals[val_mask]
        y_val  = labels[val_mask]
        X_test = signals[test_mask]
        y_test = labels[test_mask]
    else:
        raise KeyError("NPZ must contain X_val/X_test or signals/labels/splits")

    X_val  = np.asarray(X_val, dtype=np.float32)
    y_val  = np.asarray(y_val, dtype=np.int64)
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.int64)

    print(f"Val:  {X_val.shape}, Test: {X_test.shape}")
    return X_val, y_val, X_test, y_test


def load_model_from_config(cfg: Dict, device: str) -> nn.Module:
    ckpt_path = Path(cfg["checkpoint"])
    if not ckpt_path.is_absolute():
        ckpt_path = PROJECT_ROOT / ckpt_path
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

    module = importlib.import_module(cfg["module"])
    factory = getattr(module, cfg["class"])
    model = factory(**cfg["kwargs"])

    state = torch.load(str(ckpt_path), map_location=device, weights_only=False)
    if isinstance(state, dict):
        preferred_key = cfg.get("state_key")
        if preferred_key and preferred_key in state:
            state = state[preferred_key]
        else:
            for key in ("model_state_dict", "state_dict", "model_state", "model", "ema_state_dict"):
                if key in state:
                    state = state[key]
                    break

    if isinstance(state, dict):
        for prefix in ("module.", "_orig_mod."):
            if state and all(isinstance(k, str) and k.startswith(prefix) for k in state.keys()):
                state = {k[len(prefix):]: v for k, v in state.items()}

    model.load_state_dict(state, strict=cfg.get("strict", True))
    model.to(device)
    model.eval()
    return model


In [5]:
@torch.no_grad()
def get_probabilities(model: nn.Module, X: np.ndarray, device: str, batch_size: int = 64) -> np.ndarray:
    model.eval()
    all_probs = []
    X = np.asarray(X, dtype=np.float32)

    iterator = range(0, len(X), batch_size)
    for i in tqdm(iterator, total=(len(X) + batch_size - 1) // batch_size, leave=False):
        batch = torch.from_numpy(X[i:i + batch_size]).to(device, non_blocking=True)
        logits = model(batch)
        if isinstance(logits, tuple):
            logits = logits[0]
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_probs.append(probs)

    return np.concatenate(all_probs, axis=0)


def find_optimal_threshold_on_val(probs_val: np.ndarray, y_val: np.ndarray) -> float:
    best_thr, best_f1 = 0.5, 0.0
    for thr in np.arange(0.05, 0.96, 0.02):
        preds = (probs_val >= thr).astype(int)
        f1 = f1_score(y_val, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr
    return float(best_thr)


def _safe_roc_auc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    try:
        return float(roc_auc_score(y_true, y_score))
    except ValueError:
        return float("nan")


def compute_metrics(probs: np.ndarray, targets: np.ndarray, threshold: float) -> Dict:
    preds = (probs >= threshold).astype(int)

    try:
        macro_auc = float(roc_auc_score(targets, probs, average="macro"))
    except ValueError:
        macro_auc = float("nan")

    try:
        macro_auprc = float(average_precision_score(targets, probs, average="macro"))
    except ValueError:
        macro_auprc = float("nan")

    metrics = {
        "macro_AUROC": round(macro_auc, 4),
        "macro_AUPRC": round(macro_auprc, 4),
        "macro_F1": round(f1_score(targets, preds, average="macro", zero_division=0), 4),
        "threshold": round(threshold, 2),
    }

    for i, cls in enumerate(CLASS_NAMES):
        metrics[f"AUROC_{cls}"] = round(_safe_roc_auc(targets[:, i], probs[:, i]), 4)

    return metrics


In [6]:
# Fast sanity check before long inference.
device = resolve_device(DEVICE)
print("Device:", device)

for model_name, cfg in MODELS_CONFIG.items():
    if not cfg["enabled"]:
        print(f"[SKIP] {model_name}")
        continue
    ckpt_path = PROJECT_ROOT / cfg["checkpoint"]
    print(f"[CHECK] {model_name}: checkpoint={ckpt_path.exists()} module={cfg['module']} class={cfg['class']}")
    importlib.import_module(cfg["module"])


Device: cuda
[SKIP] Channel_Attention
[CHECK] LeadWise_GNN: checkpoint=True module=src.models.model_Dimash.leadwise_gnn class=LeadWiseResNetGNN
[CHECK] InceptionTime: checkpoint=True module=src.models.model_baseline.ecg_inceptiontime class=InceptionTimeBaseline
[SKIP] ResNet1D_Wang
[CHECK] RetNet: checkpoint=True module=src.models.model_Nurik.ecg_retnet class=RetNetECG


In [7]:
def run_evaluation(data_path=DATA_PATH, device=DEVICE, default_batch_size=DEFAULT_BATCH_SIZE):
    device = resolve_device(device)
    print(f"Device: {device}")

    X_val, y_val, X_test, y_test = load_data(str(data_path))
    results = {}

    for model_name, cfg in MODELS_CONFIG.items():
        if not cfg["enabled"]:
            print(f"\n[SKIP] {model_name} - disabled in config")
            continue

        batch_size = int(cfg.get("batch_size", default_batch_size))
        print(f"\n{'=' * 55}")
        print(f"  Evaluating: {model_name}  (batch_size={batch_size})")
        print(f"{'=' * 55}")

        try:
            model = load_model_from_config(cfg, device)
        except Exception as e:
            print(f"  [ERROR] Could not load model: {e}")
            continue

        print("  -> Inference on val set...")
        probs_val = get_probabilities(model, X_val, device, batch_size)

        threshold = find_optimal_threshold_on_val(probs_val, y_val)
        print(f"  -> Optimal threshold (from val): {threshold:.2f}")

        print("  -> Inference on test set...")
        probs_test = get_probabilities(model, X_test, device, batch_size)

        metrics = compute_metrics(probs_test, y_test, threshold)
        results[model_name] = metrics

        print(f"  macro-AUROC : {metrics['macro_AUROC']}")
        print(f"  macro-AUPRC : {metrics['macro_AUPRC']}")
        print(f"  macro-F1    : {metrics['macro_F1']}  (thr={metrics['threshold']})")
        print("  Per-class AUROC:")
        for cls in CLASS_NAMES:
            print(f"    {cls:6s}: {metrics[f'AUROC_{cls}']}")

        del model
        if device == "cuda":
            torch.cuda.empty_cache()

    if results:
        print(f"\n{'=' * 55}")
        print("  FINAL COMPARISON TABLE")
        print(f"{'=' * 55}")
        print(f"  {'Model':<22} {'AUROC':>7} {'AUPRC':>7} {'F1':>7} {'Thr':>5}")
        print("  " + "-" * 50)
        for name, m in results.items():
            print(f"  {name:<22} {m['macro_AUROC']:>7.4f} {m['macro_AUPRC']:>7.4f} {m['macro_F1']:>7.4f} {m['threshold']:>5.2f}")

        RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
        cols = ["model", "macro_AUROC", "macro_AUPRC", "macro_F1", "threshold"] + [f"AUROC_{c}" for c in CLASS_NAMES]
        with open(RESULTS_CSV, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=cols)
            writer.writeheader()
            for name, m in results.items():
                writer.writerow({"model": name, **m})
        print(f"\nSaved -> {RESULTS_CSV}")

    return results


In [9]:
import os, sys, importlib
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/ecg-diploma-main 3")
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import eval_final
eval_final = importlib.reload(eval_final)

MODELS_CONFIG = eval_final.MODELS_CONFIG
CLASS_NAMES = eval_final.CLASS_NAMES

print("eval_final:", eval_final.__file__)
print("enabled:", [name for name, cfg in MODELS_CONFIG.items() if cfg["enabled"]])


eval_final: /content/drive/MyDrive/ecg-diploma-main 3/eval_final.py
enabled: ['Model_A_Baseline', 'Channel_Attention', 'LeadWise_GNN', 'InceptionTime', 'RetNet']


In [10]:
results = run_evaluation()


Device: cuda
Val:  (2183, 12, 1000), Test: (2198, 12, 1000)

  Evaluating: Model_A_Baseline  (batch_size=64)
  -> Inference on val set...


  0%|          | 0/35 [00:00<?, ?it/s]

  -> Optimal threshold (from val): 0.67
  -> Inference on test set...


  0%|          | 0/35 [00:00<?, ?it/s]

  macro-AUROC : 0.9075
  macro-AUPRC : 0.7721
  macro-F1    : 0.6888  (thr=0.67)
  Per-class AUROC:
    NORM  : 0.9379
    MI    : 0.9147
    STTC  : 0.9284
    CD    : 0.9148
    HYP   : 0.8416

  Evaluating: Channel_Attention  (batch_size=64)
  -> Inference on val set...


...

  macro-AUROC : 0.9067
  macro-AUPRC : 0.7643
  macro-F1    : 0.6954  (thr=0.61)
  Per-class AUROC:
    NORM  : 0.9347
    MI    : 0.9167
    STTC  : 0.9237
    CD    : 0.9144
    HYP   : 0.8441

  Evaluating: RetNet  (batch_size=16)
  -> Inference on val set...


  0%|          | 0/137 [00:00<?, ?it/s]

  -> Optimal threshold (from val): 0.57
  -> Inference on test set...


  0%|          | 0/138 [00:00<?, ?it/s]

  macro-AUROC : 0.9132
  macro-AUPRC : 0.7843
  macro-F1    : 0.7242  (thr=0.57)
  Per-class AUROC:
    NORM  : 0.9428
    MI    : 0.9289
    STTC  : 0.9374
    CD    : 0.9242
    HYP   : 0.8327

  FINAL COMPARISON TABLE
  Model                    AUROC   AUPRC      F1   Thr
  --------------------------------------------------
  Model_A_Baseline        0.9075  0.7721  0.6888  0.67
  Channel_Attention       0.9138  0.7913  0.6993  0.71
  LeadWise_GNN            0.9086  0.7804  0.7100  0.53
  InceptionTime           0.9067  0.7643  0.6954  0.61
  RetNet                  0.9132  0.7843  0.7242  0.57

Saved -> /content/drive/MyDrive/ecg-diploma-main 3/results/eval_final.csv
